# Solution: Diagnose a Tumor — Compare 3 Classifiers

Predict whether a tumor is **malignant (`0`)** or **benign (`1`)** from cell-nucleus measurements, using three models:
**Logistic Regression**, **SVM**, and a **Decision Tree** — each with a classification report.

## Step 1 — Load the data

In [1]:
import pandas as pd

df = pd.read_csv('breast_cancer.csv')
print('shape:', df.shape)
df.head()

shape: (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## Step 2 — Get to know the data

In [2]:
print('class balance (0=malignant, 1=benign):')
print(df['target'].value_counts(normalize=True).round(3))
print('\nmissing values:', df.isnull().sum().sum())

class balance (0=malignant, 1=benign):
target
1    0.627
0    0.373
Name: proportion, dtype: float64

missing values: 0


## Step 3 — Features (X) and target (y)

In [3]:
X = df.drop(columns='target')
y = df['target']
print('features:', X.shape[1])

features: 30


## Step 4 — Train / test split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print('train:', X_train.shape[0], ' test:', X_test.shape[0])

train: 455  test: 114


## Step 5 — Scale the features (for Logistic Regression & SVM)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 6 — Model 1: Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_train_scaled, y_train)

print('Logistic Regression accuracy:', round(logreg.score(X_test_scaled, y_test), 3))
print(classification_report(y_test, logreg.predict(X_test_scaled),
                            target_names=['malignant', 'benign']))

Logistic Regression accuracy: 0.982
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## Step 7 — Model 2: Support Vector Machine

In [7]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf')
svm.fit(X_train_scaled, y_train)

print('SVM accuracy:', round(svm.score(X_test_scaled, y_test), 3))
print(classification_report(y_test, svm.predict(X_test_scaled),
                            target_names=['malignant', 'benign']))

SVM accuracy: 0.982
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## Step 8 — Model 3: Decision Tree (unscaled data)

In [8]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)   # note: original, unscaled features

print('Decision Tree — train accuracy:', round(tree.score(X_train, y_train), 3))
print('Decision Tree — test accuracy: ', round(tree.score(X_test, y_test), 3))
print(classification_report(y_test, tree.predict(X_test),
                            target_names=['malignant', 'benign']))

Decision Tree — train accuracy: 1.0
Decision Tree — test accuracy:  0.912
              precision    recall  f1-score   support

   malignant       0.85      0.93      0.89        42
      benign       0.96      0.90      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.92      0.91       114
weighted avg       0.92      0.91      0.91       114



## Step 9 — Compare

1. **Best overall accuracy:** Logistic Regression and SVM tie at the top (~0.98), well ahead of the Decision Tree (~0.91).
2. **Best recall on `malignant`:** Logistic Regression / SVM again — they miss very few malignant tumors, which is exactly what matters in a medical screen.
3. **Decision Tree:** its **training accuracy is 1.0** but test accuracy is only ~0.91 — a clear sign of **overfitting**. The unconstrained tree memorized the training data. Limiting its depth (see the bonus) closes that gap.

## Bonus — a shallower tree and a random forest

In [9]:
from sklearn.ensemble import RandomForestClassifier

# a shallower tree overfits less
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print('Decision Tree (max_depth=3) test accuracy:', round(tree3.score(X_test, y_test), 3))

# a random forest usually beats a single tree
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
print('Random Forest test accuracy:          ', round(rf.score(X_test, y_test), 3))

Decision Tree (max_depth=3) test accuracy: 0.939


Random Forest test accuracy:           0.956


## Takeaways

- The **same workflow** (split → [scale] → fit → classification report) works for all three models — only the model line changes.
- **Logistic Regression** and **SVM** benefit from scaling; the **Decision Tree** doesn't need it.
- A default **Decision Tree overfits** (train accuracy 1.0). Capping `max_depth` or using a **Random Forest** fixes most of that.
- Always read the **classification report**, not just accuracy — in a medical setting, recall on the dangerous class is what counts.